# CliniScan - Detection Model 

In [ ]:
import os,random,shutil,warnings,json
warnings.filterwarnings('ignore')
import numpy as np,pandas as pd,cv2,yaml
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.patches as patches
from tqdm import tqdm; import torch, time

ENV='local'  # 'local' | 'kaggle' | 'colab'
if ENV=='local':   BASE_DIR=r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan"
elif ENV=='kaggle': BASE_DIR="/kaggle/working/CliniScan"
else:               BASE_DIR="/content/CliniScan"

IMAGE_DIR=os.path.join(BASE_DIR,'images3000_processed')
LABEL_DIR=os.path.join(BASE_DIR,'yolo_labels')
CSV_PATH=os.path.join(BASE_DIR,'train.csv')
OUTPUT_DIR=os.path.join(BASE_DIR,'yolo_output')
DATASET_DIR=os.path.join(BASE_DIR,'dataset')
M3_YOLO_DIR=os.path.join(BASE_DIR,'milestone3_det')
TRAIN_IMG_DIR=os.path.join(DATASET_DIR,'images','train')
VAL_IMG_DIR  =os.path.join(DATASET_DIR,'images','val')
TEST_IMG_DIR =os.path.join(DATASET_DIR,'images','test')
TRAIN_LABEL_DIR=os.path.join(DATASET_DIR,'labels','train')
VAL_LABEL_DIR  =os.path.join(DATASET_DIR,'labels','val')
TEST_LABEL_DIR =os.path.join(DATASET_DIR,'labels','test')
for d in [OUTPUT_DIR,M3_YOLO_DIR,M3_YOLO_DIR+"/plots"]: os.makedirs(d,exist_ok=True)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_DEVICE=0 if torch.cuda.is_available() else 'cpu'
GPU_BATCH=16 if torch.cuda.is_available() else 8
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark=True; print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    torch.set_num_threads(4); print("CPU (threads=4)")

IMG_SIZE=224; EPOCHS=30; BATCH_SIZE=GPU_BATCH; SEED=42
CLASS_NAMES=['Aortic enlargement','Atelectasis','Calcification','Cardiomegaly',
    'Consolidation','ILD','Infiltration','Lung Opacity','Nodule/Mass','Other lesion',
    'Pleural effusion','Pleural thickening','Pneumothorax','Pulmonary fibrosis']
NUM_CLASSES=len(CLASS_NAMES)
COLORS=plt.cm.tab20(np.linspace(0,1,NUM_CLASSES))
random.seed(SEED); np.random.seed(SEED)
DET_BASELINE=dict(map50=0.0658,map50_95=0.0286,precision=0.3793,recall=0.0821)
print(f"ENV={ENV} | Device={DEVICE} | Batch={GPU_BATCH}")
print(f"Images: {'OK' if os.path.exists(IMAGE_DIR) else 'MISSING'}")
print(f"dataset: {'OK' if os.path.exists(DATASET_DIR) else 'MISSING'}")


In [ ]:
os.makedirs(LABEL_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print(f'CSV rows total       : {len(df)}')

# Only process images available locally
available = set(f.replace('.png','') for f in os.listdir(IMAGE_DIR) if f.endswith('.png'))
df = df[df['image_id'].isin(available)]
print(f'Images found locally : {len(available)}')
print(f'CSV rows matched     : {len(df)}')

generated = 0
skipped   = 0

for image_id, group in tqdm(df.groupby('image_id'), desc='Generating YOLO labels'):
    img_path = os.path.join(IMAGE_DIR, image_id + '.png')
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        skipped += 1
        continue

    orig_h, orig_w = img.shape[:2]  # 224x224 for your images
    yolo_lines = []

    # Get rows with actual bounding boxes for this image
    img_rows_with_box = group[~group['x_max'].isna() & (group['class_id'] != 14)]

    for _, row in group.iterrows():
        class_id = int(row['class_id'])
        if class_id == 14:           # skip No finding
            continue
        if pd.isna(row['x_min']):    # skip rows with no bbox
            continue

        x_min = float(row['x_min'])
        y_min = float(row['y_min'])
        x_max = float(row['x_max'])
        y_max = float(row['y_max'])

        # Estimate original DICOM size from max bbox coords in this image
        # VinBigData annotations are in original DICOM pixel space
        if len(img_rows_with_box) > 0:
            orig_dicom_w = img_rows_with_box['x_max'].max() * 1.05
            orig_dicom_h = img_rows_with_box['y_max'].max() * 1.05
        else:
            orig_dicom_w = 3000
            orig_dicom_h = 3000

        # Scale bbox from original DICOM coords to 224x224
        scale_x = orig_w / orig_dicom_w
        scale_y = orig_h / orig_dicom_h

        x_min_s = max(0, x_min * scale_x)
        y_min_s = max(0, y_min * scale_y)
        x_max_s = min(orig_w, x_max * scale_x)
        y_max_s = min(orig_h, y_max * scale_y)

        if x_max_s <= x_min_s or y_max_s <= y_min_s:
            continue

        # Convert to YOLO format — normalized 0 to 1
        xc = ((x_min_s + x_max_s) / 2) / orig_w
        yc = ((y_min_s + y_max_s) / 2) / orig_h
        bw = (x_max_s - x_min_s) / orig_w
        bh = (y_max_s - y_min_s) / orig_h

        # Clamp all to [0, 1]
        xc = min(max(xc, 0), 1)
        yc = min(max(yc, 0), 1)
        bw = min(max(bw, 0), 1)
        bh = min(max(bh, 0), 1)

        yolo_lines.append(f'{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

    # Save label file — empty file = normal/no finding image
    label_path = os.path.join(LABEL_DIR, image_id + '.txt')
    with open(label_path, 'w') as f:
        f.write('\n'.join(yolo_lines))
    generated += 1

# Empty label for images not in CSV at all
for img_id in available:
    label_path = os.path.join(LABEL_DIR, img_id + '.txt')
    if not os.path.exists(label_path):
        open(label_path, 'w').close()

total_labels = len([f for f in os.listdir(LABEL_DIR) if f.endswith('.txt')])
print(f'\n=== YOLO Label Generation Complete ===')
print(f'  Labels generated : {generated}')
print(f'  Images skipped   : {skipped}')
print(f'  Total .txt files : {total_labels}')
print(f'  Saved to         : {LABEL_DIR}')

In [ ]:
# Get all valid image-label pairs
valid_pairs = []
missing_labels = []
empty_labels   = []

for fname in os.listdir(IMAGE_DIR):
    if not fname.endswith('.png'):
        continue
    label_name = fname.replace('.png', '.txt')
    label_path = os.path.join(LABEL_DIR, label_name)

    if not os.path.exists(label_path):
        missing_labels.append(fname)
    else:
        valid_pairs.append(fname)
        with open(label_path) as f:
            content = f.read().strip()
        if content == '':
            empty_labels.append(fname)   # normal images — no finding

print(f"Total images          : {png_count}")
print(f"Valid pairs           : {len(valid_pairs)}")
print(f"Missing label files   : {len(missing_labels)}")
print(f"Empty labels (normal) : {len(empty_labels)}")
print(f"Images with findings  : {len(valid_pairs) - len(empty_labels)}")

# Verify YOLO format on 5 random label files
print("\nSample YOLO label check:")
sample_labels = random.sample([f for f in valid_pairs if f not in empty_labels], min(5, len(valid_pairs)))
for fname in sample_labels:
    label_path = os.path.join(LABEL_DIR, fname.replace('.png', '.txt'))
    with open(label_path) as f:
        lines = f.readlines()
    print(f"  {fname[:30]} → {len(lines)} boxes | first: {lines[0].strip() if lines else 'empty'}")

print("\n✅ YOLO annotation format verified")

In [ ]:
# Pick 10 images that have at least one bounding box
images_with_boxes = [f for f in valid_pairs if f not in empty_labels]
sample_10 = random.sample(images_with_boxes, min(10, len(images_with_boxes)))

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

COLORS = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

for i, fname in enumerate(sample_10):
    img_path   = os.path.join(IMAGE_DIR, fname)
    label_path = os.path.join(LABEL_DIR, fname.replace('.png', '.txt'))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    axes[i].imshow(img, cmap='gray')

    with open(label_path) as f:
        lines = f.readlines()

    found_classes = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:])

        # Convert YOLO → pixel coords
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        box_w = int(bw * w)
        box_h = int(bh * h)

        color = COLORS[cls_id % NUM_CLASSES]
        rect  = patches.Rectangle((x1, y1), box_w, box_h,
                                   linewidth=2, edgecolor=color, facecolor='none')
        axes[i].add_patch(rect)
        axes[i].text(x1, max(y1-3, 0), CLASS_NAMES[cls_id],
                     fontsize=6, color='white',
                     bbox=dict(facecolor=color, alpha=0.7, pad=1))
        found_classes.append(CLASS_NAMES[cls_id])

    axes[i].set_title(f"{len(lines)} boxes", fontsize=8)
    axes[i].axis('off')

plt.suptitle('Data Verification — 10 Random Samples with Bounding Boxes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'bbox_verification.png'), dpi=100)
plt.show()
print("✅ Bounding box verification complete")

In [ ]:
random.shuffle(valid_pairs)
n = len(valid_pairs)

train_files = valid_pairs[ : int(0.70 * n)]
val_files   = valid_pairs[int(0.70 * n) : int(0.85 * n)]
test_files  = valid_pairs[int(0.85 * n) : ]

print(f'Total  : {n}')
print(f'Train  : {len(train_files)}  ({len(train_files)/n*100:.1f}%)')
print(f'Val    : {len(val_files)}   ({len(val_files)/n*100:.1f}%)')
print(f'Test   : {len(test_files)}  ({len(test_files)/n*100:.1f}%)')

# ── Build mandatory folder structure from document §3 ──────
# dataset/
#   images/train/   images/val/   images/test/
#   labels/train/   labels/val/   labels/test/
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

for d in [TRAIN_IMG_DIR, VAL_IMG_DIR, TEST_IMG_DIR,
          TRAIN_LABEL_DIR, VAL_LABEL_DIR, TEST_LABEL_DIR]:
    os.makedirs(d, exist_ok=True)

split_map = [
    (train_files, TRAIN_IMG_DIR, TRAIN_LABEL_DIR),
    (val_files,   VAL_IMG_DIR,   VAL_LABEL_DIR),
    (test_files,  TEST_IMG_DIR,  TEST_LABEL_DIR),
]

for files, img_dst, lbl_dst in split_map:
    split_name = os.path.basename(img_dst)
    for fname in tqdm(files, desc=f'Copying {split_name}'):
        shutil.copy(os.path.join(IMAGE_DIR, fname),
                    os.path.join(img_dst, fname))
        label_name = fname.replace('.png', '.txt')
        shutil.copy(os.path.join(LABEL_DIR, label_name),
                    os.path.join(lbl_dst, label_name))

print('\n✅ Mandatory folder structure created (document §3):')
print(f'   dataset/')
print(f'   ├── images/train/  ({len(train_files)} images)')
print(f'   ├── images/val/    ({len(val_files)} images)')
print(f'   ├── images/test/   ({len(test_files)} images)')
print(f'   ├── labels/train/')
print(f'   ├── labels/val/')
print(f'   └── labels/test/')


In [ ]:
yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')

# Document §6 YAML format
yaml_content = {
    'path'  : DATASET_DIR.replace('\\\\', '/').replace('\\', '/'),
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : NUM_CLASSES,
    'names' : CLASS_NAMES
}

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print('✅ dataset.yaml created (document §6 format):')
print(f'   {yaml_path}')
print('\nContents:')
with open(yaml_path) as f:
    print(f.read())


In [ ]:
from ultralytics import YOLO

# yolov8s = small — document §4 recommends yolov8s
# Slightly heavier than nano but better accuracy
model = YOLO('yolov8s.pt')   # auto-downloads ~22MB

print('✅ YOLOv8s loaded (document §4: primary model yolov8s)')
print()
print('Safe medical augmentations (document §5):')
print('  ✅ Horizontal Flip      — allowed')
print('  ✅ Small Rotation ±10°  — allowed')
print('  ✅ Brightness/Contrast  — allowed')
print('  ✅ Mild Gaussian Noise  — allowed')
print('  ✅ Random Scaling       — allowed')
print('  ❌ Vertical Flip        — NOT allowed')
print('  ❌ Large Rotations      — NOT allowed')
print('  ❌ Perspective Warping  — NOT allowed')
print('  ❌ Color Transformations— NOT allowed')
print('  ❌ Mosaic/Mixup         — NOT allowed (medical realism)')


In [ ]:
import torch, os

# Max CPU speed
torch.set_num_threads(4)
torch.set_num_interop_threads(2)
os.environ['OMP_NUM_THREADS']      = '4'
os.environ['MKL_NUM_THREADS']      = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
print(f"CPU threads : {torch.get_num_threads()}")

print('Starting YOLOv8s training...')
print(f'  Image size : 224')
print(f'  Epochs     : 30')
print(f'  Batch size : 8')
print(f'  Cache      : OFF  (low RAM — 0.9GB free)')
print('-' * 50)

results = model.train(
    # ── Dataset ──────────────────────────────────────
    data        = yaml_path,
    imgsz       = 224,

    # ── Training ─────────────────────────────────────
    epochs      = 30,
    batch       = 8,
    optimizer   = 'Adam',
    lr0         = 0.001,
    lrf         = 0.01,
    iou         = 0.5,
    conf        = 0.25,

    # ── Device ───────────────────────────────────────
    device      = 'cpu',
    workers     = 0,

    # ── Speed optimizations (RAM safe) ───────────────
    cache       = False,          # OFF — only 0.9GB free
    rect        = True,           # ~15% faster, no RAM cost
    cos_lr      = True,           # smarter LR scheduling
    amp         = True,           # faster math, saves RAM
    nbs         = 64,

    # ── Safe medical augmentations ────────────────────
    fliplr      = 0.5,            # ✅ horizontal flip
    flipud      = 0.0,            # ❌ vertical flip
    degrees     = 10.0,           # ✅ rotation ±10°
    scale       = 0.1,            # ✅ random scaling
    hsv_v       = 0.1,            # ✅ brightness
    hsv_h       = 0.0,            # ❌ no hue
    hsv_s       = 0.0,            # ❌ no saturation
    perspective = 0.0,            # ❌ no warp
    erasing     = 0.0,            # ❌ no erasing
    mosaic      = 0.0,            # ❌ disabled
    mixup       = 0.0,            # ❌ disabled
    copy_paste  = 0.0,            # ❌ disabled

    # ── Saving ───────────────────────────────────────
    project     = OUTPUT_DIR,
    name        = 'yolov8s_cliniScan',
    save        = True,
    save_period = 1,              # checkpoint every epoch
    plots       = True,
    verbose     = True,
)

print('\n✅ YOLOv8s training complete!')

In [ ]:
from ultralytics import YOLO
import os, torch

torch.set_num_threads(4)
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

last_pt = r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan\yolo_output\yolov8s_cliniScan\weights\last.pt"

print(f"last.pt exists : {os.path.exists(last_pt)}")
print("Resuming from epoch 15 — 14 epochs remaining...")

model_resume = YOLO(last_pt)
results = model_resume.train(resume=True)

print("✅ Training complete!")

In [ ]:
from ultralytics import YOLO
import os, torch

torch.set_num_threads(4)
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

# Paths
best_pt   = r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan\yolo_output\yolov8s_cliniScan\weights\best.pt"
yaml_path = r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan\dataset\dataset.yaml"

print(f"best.pt exists : {os.path.exists(best_pt)}")
print(f"yaml exists    : {os.path.exists(yaml_path)}")
print("Starting 10 more epochs from best model...")
print("-" * 50)

model_continue = YOLO(best_pt)

results = model_continue.train(
    data        = yaml_path,
    imgsz       = 224,
    epochs      = 10,
    batch       = 8,
    optimizer   = 'Adam',
    lr0         = 0.0001,        # lower LR — fine tuning
    lrf         = 0.01,
    iou         = 0.5,
    conf        = 0.25,
    device      = 'cpu',
    workers     = 0,
    cache       = False,
    rect        = True,
    cos_lr      = True,
    amp         = True,
    nbs         = 64,
    fliplr      = 0.5,
    flipud      = 0.0,
    degrees     = 10.0,
    scale       = 0.1,
    hsv_v       = 0.1,
    hsv_h       = 0.0,
    hsv_s       = 0.0,
    perspective = 0.0,
    erasing     = 0.0,
    mosaic      = 0.0,
    mixup       = 0.0,
    copy_paste  = 0.0,
    project     = r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan\yolo_output",
    name        = 'yolov8s_cliniScan_v2',
    save        = True,
    save_period = 1,
    plots       = True,
    verbose     = True,
)

print("\n✅ 10 extra epochs complete!")

# ── Auto compare both models after training ───────────────
print("\nComparing both models...")
print("-" * 50)

m1 = YOLO(r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan\yolo_output\yolov8s_cliniScan\weights\best.pt")
m2 = YOLO(r"C:\Users\Anshuman Sharma\Desktop\Project\B13-CliniScan\yolo_output\yolov8s_cliniScan_v2\weights\best.pt")

r1 = m1.val(data=yaml_path, verbose=False)
r2 = m2.val(data=yaml_path, verbose=False)

print(f"\n  30 epochs  mAP50 : {r1.box.map50:.4f}")
print(f"  40 epochs  mAP50 : {r2.box.map50:.4f}")

if r2.box.map50 > r1.box.map50:
    print(f"\n  ✅ 40 epochs model is BETTER — use yolov8s_cliniScan_v2/weights/best.pt")
else:
    print(f"\n  ✅ 30 epochs model is BETTER — use yolov8s_cliniScan/weights/best.pt")

In [ ]:
import matplotlib.image as mpimg

results_dir = os.path.join(OUTPUT_DIR, 'yolov8s_cliniScan')

# YOLOv8 auto-saves these — document §10 deliverables
plot_files = {
    'results.png'          : 'Training Losses + mAP Curves',
    'confusion_matrix.png' : 'Confusion Matrix',
    'PR_curve.png'         : 'Precision-Recall Curve  (document §10)',
    'F1_curve.png'         : 'F1 Score Curve',
}

for plot_file, title in plot_files.items():
    plot_path = os.path.join(results_dir, plot_file)
    if os.path.exists(plot_path):
        img = mpimg.imread(plot_path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.title(title, fontsize=13, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print(f'✅ {title}')
    else:
        print(f'⚠️  {plot_file} not found — run after training completes')


In [ ]:
from ultralytics import YOLO
import os
best_weights = os.path.join(OUTPUT_DIR, 'yolov8s_cliniScan', 'weights', 'best.pt')
best_model   = YOLO(best_weights)

print('Running validation on val set...')
metrics = best_model.val(data=yaml_path, verbose=True)

print('\n' + '='*55)
print('   VALIDATION RESULTS  (document §7 metrics)')
print('='*55)
print(f'   mAP@50        : {metrics.box.map50:.4f}   (main metric)')
print(f'   mAP@50-95     : {metrics.box.map:.4f}   (COCO strict)')
print(f'   Precision     : {metrics.box.mp:.4f}')
print(f'   Recall        : {metrics.box.mr:.4f}')
print('='*55)
print()

# Document §9 — Validation Checklist
print('VALIDATION CHECKLIST (document §9):')
print(f'  [{"✔" if metrics.box.map50 > 0.0 else "✘"}] mAP improved from baseline')
print(f'  [{"✔" if metrics.box.mp > 0.3 else "check"}] No excessive false positives  (Precision={metrics.box.mp:.3f})')
print(f'  [{"✔" if metrics.box.mr > 0.2 else "check"}] No major missed abnormalities (Recall={metrics.box.mr:.3f})')
print( '  [check manually] Bounding boxes tightly aligned')
print(f'  [{"✔" if metrics.box.map50 > 0.0 else "check"}] Confidence scores reasonable')
print( '  [✔] Logs and metrics saved by YOLOv8 automatically')


In [ ]:
# Document §10: sample detection visualizations 20-30 images
test_img_dir = TEST_IMG_DIR
test_images  = os.listdir(test_img_dir)
sample_test  = random.sample(test_images, min(20, len(test_images)))

print(f'Running predictions on {len(sample_test)} test images...')

# Save all predictions as grid — 4 per row
cols = 4
rows = (len(sample_test) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5))
axes = axes.flatten()

detection_summary = []

for i, fname in enumerate(sample_test):
    img_path = os.path.join(test_img_dir, fname)
    result   = best_model.predict(img_path, conf=0.25, verbose=False)[0]

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    axes[i].imshow(img)
    num_boxes = 0
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        color    = COLORS[cls_id % NUM_CLASSES]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=color, facecolor='none')
        axes[i].add_patch(rect)
        axes[i].text(x1, max(y1-3, 0),
                     f'{CLASS_NAMES[cls_id]} {conf_val:.2f}',
                     fontsize=6, color='white',
                     bbox=dict(facecolor=color, alpha=0.8, pad=1))
        num_boxes += 1
        detection_summary.append({'image': fname, 'class': CLASS_NAMES[cls_id], 'conf': conf_val})

    title = f'{num_boxes} finding(s)' if num_boxes > 0 else 'Normal'
    axes[i].set_title(title, fontsize=8, color='red' if num_boxes > 0 else 'green')
    axes[i].axis('off')

# Hide unused axes
for j in range(len(sample_test), len(axes)):
    axes[j].axis('off')

plt.suptitle('YOLOv8s — 20 Test Image Predictions (document §10)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'test_predictions_20.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'✅ 20 test predictions saved → {save_path}')

# Detection summary table
if detection_summary:
    df_det = pd.DataFrame(detection_summary)
    print(f'\nDetection summary ({len(detection_summary)} total detections):')
    print(df_det.groupby('class')['conf'].agg(['count','mean']).round(3).to_string())


In [ ]:
import os, shutil

# ── Recalculate file counts from dataset folder ───────────
train_count = len(os.listdir(os.path.join(DATASET_DIR, 'images', 'train')))
val_count   = len(os.listdir(os.path.join(DATASET_DIR, 'images', 'val')))
test_count  = len(os.listdir(os.path.join(DATASET_DIR, 'images', 'test')))

# ── Copy best model ───────────────────────────────────────
# Use v2 if exists, otherwise v1
v2 = os.path.join(OUTPUT_DIR, 'yolov8s_cliniScan_v2', 'weights', 'best.pt')
v1 = os.path.join(OUTPUT_DIR, 'yolov8s_cliniScan',    'weights', 'best.pt')
best_src         = v2 if os.path.exists(v2) else v1
final_model_path = os.path.join(OUTPUT_DIR, 'best.pt')
shutil.copy(best_src, final_model_path)

# ── Results Table- 30 + 10 epochs trained ────────────────────────────────────────
print('='*60)
print('   CLINI-SCAN DETECTION — FINAL RESULTS TABLE')
print('='*60)
print(f'   Model          : YOLOv8s')
print(f'   Image size     : 224x224')
print(f'   Epochs trained : 40  (30 + 10 fine-tuning)')
print(f'   Training imgs  : {train_count}')
print(f'   Val imgs       : {val_count}')
print(f'   Test imgs      : {test_count}')
print(f'   Classes        : {NUM_CLASSES}')
print()
print(f'   mAP@50         : {metrics.box.map50:.4f}')
print(f'   mAP@50-95      : {metrics.box.map:.4f}')
print(f'   Precision      : {metrics.box.mp:.4f}')
print(f'   Recall         : {metrics.box.mr:.4f}')
print('='*60)

# ── Summary Report ────────────────────────────────────────
report = f"""CliniScan — Object Detection Model Summary Report
=================================================
Model Used       : YOLOv8s (pretrained on COCO)
Task             : Multi-class bounding box detection
Dataset          : VinBigData Chest X-ray (5000 images subset)
Classes          : {NUM_CLASSES} abnormality types

Hyperparameters
---------------
Image size       : {IMG_SIZE}x{IMG_SIZE}
Epochs           : {EPOCHS}
Batch size       : {BATCH_SIZE}
Optimizer        : Adam
Learning rate    : 0.001
IoU threshold    : 0.5
Conf threshold   : 0.25

Augmentations Applied (Medical Safe)
-------------------------------------
Horizontal flip  : Yes (p=0.5)
Rotation         : Yes (±10 degrees)
Brightness adj   : Yes (±10%)
Random scaling   : Yes (10%)
Vertical flip    : No
Mosaic/Mixup     : No
Perspective warp : No

Results
-------
mAP@50           : {metrics.box.map50:.4f}
mAP@50-95        : {metrics.box.map:.4f}
Precision        : {metrics.box.mp:.4f}
Recall           : {metrics.box.mr:.4f}

Saved Files
-----------
best.pt          : {final_model_path}
Training logs    : {os.path.join(OUTPUT_DIR, 'yolov8s_cliniScan')}
"""

report_path = os.path.join(OUTPUT_DIR, 'detection_summary_report.txt')
with open(report_path, 'w') as f:
    f.write(report)

print(report)
print(f'✅ Summary report saved → {report_path}')
print(f'✅ best.pt saved        → {final_model_path}')

---
## Predict on Any Single Image

In [ ]:
def detect_single(img_path, conf_threshold=0.25):
    """Run detection on any single chest X-ray image."""
    result = best_model.predict(img_path, conf=conf_threshold, verbose=False)[0]

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Original
    original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('Input Image', fontsize=10)
    axes[0].axis('off')

    # Predictions
    axes[1].imshow(img)
    detections = []
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        color    = COLORS[cls_id % NUM_CLASSES]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=color, facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x1, max(y1-3, 0),
                     f"{CLASS_NAMES[cls_id]} {conf_val:.2f}",
                     fontsize=8, color='white',
                     bbox=dict(facecolor=color, alpha=0.8, pad=1))
        detections.append((CLASS_NAMES[cls_id], conf_val, (x1,y1,x2,y2)))

    title = f"{len(detections)} finding(s) detected" if detections else "No finding (Normal)"
    axes[1].set_title(title, fontsize=10, color='red' if detections else 'green')
    axes[1].axis('off')

    plt.suptitle('CliniScan — Detection Result', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("Detections:")
    if detections:
        for name, conf_val, coords in sorted(detections, key=lambda x: -x[1]):
            print(f"  ✅ {name:<25} conf={conf_val:.3f}  box={coords}")
    else:
        print("  No finding (Normal)")


# Test on a random image
sample_img = os.path.join(test_img_dir, random.choice(test_images))
detect_single(sample_img)

---
# == MILESTONE 3 ==
## Start here - M2 detection already done

| Day | Task |
|---|---|
| Day 1 | D1_YOLOv8m (~3.5 hrs) |
| Day 2 | D2_LowLR + D3_LowThresh (~6 hrs) |
| Day 3 | D4_SGD (~2.5 hrs) |
| Day 4 | Tables + Visualization + Error Analysis |
| Day 5 | Review + report |

## M3 - Detection STEP A | Task 1 + 3: 4 YOLO Experiments

| # | Name | Model | LR | Conf | IoU | Optimizer | M3 Task |
|---|---|---|---|---|---|---|---|
| D1 | D1_YOLOv8m | YOLOv8m | 0.001 | 0.25 | 0.50 | AdamW | Task 3 - bigger model |
| D2 | D2_LowLR | YOLOv8s | 0.0005 | 0.25 | 0.50 | AdamW | Task 1 - LR tuning |
| D3 | D3_LowThresh | YOLOv8s | 0.001 | 0.15 | 0.45 | AdamW | Task 1 - threshold tuning |
| D4 | D4_SGD | YOLOv8s | 0.01 | 0.25 | 0.50 | SGD | Task 1 - optimizer |

**CPU: D1~3.5hrs, D2-D4~2.5hrs each = ~12hrs across 2 nights**

Set `RESUME_YOLO = True` to resume.

In [ ]:
from ultralytics import YOLO

yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')
RESUME_YOLO = True   # ← change to True every morning to resume
DET_EPOCHS  = 15

YOLO_EXPS = [
    ('D1_YOLOv8m',   'yolov8m.pt', 0.001,  0.25, 0.50, DET_EPOCHS, 'AdamW'),
    ('D2_LowLR',     'yolov8s.pt', 0.0005, 0.25, 0.50, DET_EPOCHS, 'AdamW'),
    ('D3_LowThresh', 'yolov8s.pt', 0.001,  0.15, 0.45, DET_EPOCHS, 'AdamW'),
    ('D4_SGD',       'yolov8s.pt', 0.01,   0.25, 0.50, DET_EPOCHS, 'SGD'),
]

for exp_name, model_wt, lr, conf, iou, epochs, opt_name in YOLO_EXPS:
    last_pt     = os.path.join(M3_YOLO_DIR, exp_name, 'weights', 'last.pt')
    results_csv = os.path.join(M3_YOLO_DIR, exp_name, 'results.csv')

    # ── NEW: skip if already complete ─────────────────────
    if os.path.exists(results_csv):
        import pandas as pd
        done = len(pd.read_csv(results_csv))
        if done >= epochs:
            print(f"  SKIP {exp_name} — already complete ({done} epochs)")
            continue

    # ── Resume if interrupted ──────────────────────────────
    if RESUME_YOLO and os.path.exists(last_pt):
        print(f"\n  RESUMING {exp_name} from last.pt...")
        YOLO(last_pt).train(resume=True)
        continue

    # ── Fresh start ────────────────────────────────────────
    cpu_est = epochs * (25 if 'yolov8m' in model_wt else 18)
    print(f"\n{'='*52}\n  STARTING {exp_name}")
    print(f"  model={model_wt} | lr={lr} | conf={conf} | iou={iou} | opt={opt_name}")
    print(f"  CPU est: ~{cpu_est}min (~{cpu_est/60:.1f}hrs)\n{'='*52}")

    YOLO(model_wt).train(
        data        = yaml_path,
        imgsz       = 224,
        epochs      = epochs,
        batch       = GPU_BATCH,
        optimizer   = opt_name,
        lr0         = lr,
        lrf         = 0.01,
        conf        = conf,
        iou         = iou,
        device      = GPU_DEVICE,
        workers     = 4 if torch.cuda.is_available() else 0,
        cache       = False,
        rect        = True,
        cos_lr      = True,
        amp         = True,
        fliplr      = 0.5,  flipud     = 0.0,
        degrees     = 10.0, scale      = 0.1,
        hsv_v       = 0.1,  hsv_h      = 0.0,  hsv_s = 0.0,
        perspective = 0.0,  mosaic     = 0.0,
        mixup       = 0.0,  copy_paste = 0.0,
        project     = M3_YOLO_DIR,
        name        = exp_name,
        save        = True,
        save_period = 1,    # ← changed from 5 to 1 (saves every epoch)
        plots       = True,
        verbose     = True,
    )
    print(f"  DONE {exp_name}")

In [ ]:
from ultralytics import YOLO

print("\n"+"="*75)
print("  DETECTION - M3 FULL COMPARISON TABLE (Task 4)")
print("="*75)
print(f"  {'Model':<22} {'mAP@50':<9} {'mAP50-95':<11} {'Prec':<9} {'Recall':<9} {'vs M2':<9} {'BBox Q'}")
print("-"*75)
print(f"  {'Baseline YOLOv8s M2':<22} {DET_BASELINE['map50']:<9.4f} {DET_BASELINE['map50_95']:<11.4f} {DET_BASELINE['precision']:<9.4f} {DET_BASELINE['recall']:<9.4f}  ---")

yolo_results=[]
for exp_name,model_wt,_,_,_,_,_ in YOLO_EXPS:
    wt=os.path.join(M3_YOLO_DIR,exp_name,'weights','best.pt')
    if not os.path.exists(wt): print(f"  {exp_name:<22} not found"); continue
    met=YOLO(wt).val(data=yaml_path,verbose=False)
    m50=met.box.map50; m95=met.box.map; prec=met.box.mp; rec=met.box.mr
    bbox_q=(m50+m95)/2; chg=m50-DET_BASELINE['map50']; sym='UP' if chg>0 else 'DN'
    print(f"  {exp_name:<22} {m50:<9.4f} {m95:<11.4f} {prec:<9.4f} {rec:<9.4f} {sym}{abs(chg):.4f}   {bbox_q:.4f}")
    yolo_results.append(dict(name=exp_name,model=model_wt,map50=m50,map95=m95,prec=prec,rec=rec,bbox_quality=bbox_q,vs_baseline=round(chg,4)))
print("="*75)

if yolo_results:
    best_det=max(yolo_results,key=lambda x:x['map50'])
    print(f"\n  Best: {best_det['name']} | mAP50={best_det['map50']:.4f} | improvement={best_det['vs_baseline']:+.4f}")
    pd.DataFrame(yolo_results).to_csv(M3_YOLO_DIR+"/det_results.csv",index=False)

    # mAP comparison bar chart
    fig,axes=plt.subplots(1,2,figsize=(16,6))
    names=[r['name'] for r in yolo_results]+['Baseline M2']
    m50s=[r['map50'] for r in yolo_results]+[DET_BASELINE['map50']]
    m95s=[r['map95'] for r in yolo_results]+[DET_BASELINE['map50_95']]
    colors=['#2E75B6' if m>DET_BASELINE['map50'] else '#C00000' for m in [r['map50'] for r in yolo_results]]+['#808080']
    for ax,vals,title in [(axes[0],m50s,'mAP@50'),(axes[1],m95s,'mAP@50-95')]:
        bars=ax.bar(range(len(names)),vals,color=colors,width=0.6)
        ax.axhline(DET_BASELINE['map50'],color='black',ls='--',lw=1.5,label='M2 Baseline')
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names,rotation=30,ha='right',fontsize=8)
        ax.set_title(f'{title} Comparison',fontweight='bold')
        for bar,val in zip(bars,vals): ax.text(bar.get_x()+bar.get_width()/2,val+0.001,f'{val:.4f}',ha='center',fontsize=7)
        ax.legend()
    plt.suptitle('M3 Detection Experiments vs M2 Baseline',fontsize=12,fontweight='bold')
    plt.tight_layout(); plt.savefig(M3_YOLO_DIR+"/plots/det_comparison_bar.png",dpi=100); plt.close()
    print("Detection comparison bar chart saved")


In [ ]:
from ultralytics import YOLO

if yolo_results:
    best_det_name=max(yolo_results,key=lambda x:x['map50'])['name']
    best_det_pt=os.path.join(M3_YOLO_DIR,best_det_name,'weights','best.pt')
else:
    best_det_pt=os.path.join(OUTPUT_DIR,'yolov8s_cliniScan','weights','best.pt')
    best_det_name='M2 Baseline'

m2_pt=os.path.join(OUTPUT_DIR,'yolov8s_cliniScan','weights','best.pt')
det_m3=YOLO(best_det_pt)
det_m2=YOLO(m2_pt) if os.path.exists(m2_pt) else det_m3

t_imgs=os.listdir(TEST_IMG_DIR); sample30=random.sample(t_imgs,min(30,len(t_imgs)))

correct,missed=[],[]
for fname in sample30:
    ip=os.path.join(TEST_IMG_DIR,fname)
    res=det_m3.predict(ip,conf=0.25,verbose=False)[0]
    nb=len(res.boxes) if res.boxes else 0
    vis=res.plot(); (correct if nb>0 else missed).append((vis,fname,nb))

def det_grid(cases,title,fname,n=8):
    fig,axes=plt.subplots(2,4,figsize=(22,11)); axes=axes.flatten()
    for i,(img,fn,nb) in enumerate(cases[:n]):
        axes[i].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"{fn[:18]}\n{nb} det.",fontsize=7,color='green' if nb>0 else 'red')
        axes[i].axis('off')
    for j in range(len(cases[:n]),8): axes[j].axis('off')
    plt.suptitle(title,fontsize=12,fontweight='bold'); plt.tight_layout()
    plt.savefig(os.path.join(M3_YOLO_DIR,fname),dpi=100); plt.close(); print(f"Saved: {fname}")

if correct: det_grid(correct,f'Correct Detections - {best_det_name} (M3)','correct_detections.png')
if missed:  det_grid(missed,  'Missed Detections - Failure Cases (M3)',   'missed_detections.png')

# M2 vs M3 side-by-side on 8 images
sample8=random.sample(t_imgs,min(8,len(t_imgs)))
fig,axes=plt.subplots(2,8,figsize=(32,9))
for col,fname in enumerate(sample8):
    ip=os.path.join(TEST_IMG_DIR,fname)
    r2=det_m2.predict(ip,conf=0.25,verbose=False)[0]
    r3=det_m3.predict(ip,conf=0.25,verbose=False)[0]
    n2=len(r2.boxes) if r2.boxes else 0; n3=len(r3.boxes) if r3.boxes else 0
    axes[0,col].imshow(cv2.cvtColor(r2.plot(),cv2.COLOR_BGR2RGB))
    axes[0,col].set_title(f"M2: {n2} box",fontsize=7,color='blue'); axes[0,col].axis('off')
    axes[1,col].imshow(cv2.cvtColor(r3.plot(),cv2.COLOR_BGR2RGB))
    axes[1,col].set_title(f"M3: {n3} box",fontsize=7,color='green'); axes[1,col].axis('off')
plt.suptitle('M2 vs M3 Side-by-Side on Same Test Images',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.savefig(M3_YOLO_DIR+"/plots/m2_vs_m3.png",dpi=100); plt.close()
print("M2 vs M3 comparison grid saved")
print(f"Tested:{len(sample30)} | Found:{len(correct)} | Missed:{len(missed)}")


In [ ]:
from ultralytics import YOLO

if yolo_results:
    best_det_pt2=os.path.join(M3_YOLO_DIR,best_det_name,'weights','best.pt')
    det_best=YOLO(best_det_pt2)

    # Threshold sensitivity analysis
    thresholds=[0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.50]
    thresh_res=[]
    print("Threshold sensitivity analysis:")
    for t in thresholds:
        met=det_best.val(data=yaml_path,conf=t,verbose=False)
        thresh_res.append(dict(conf=t,map50=met.box.map50,prec=met.box.mp,rec=met.box.mr))
        print(f"  conf={t:.2f} mAP50={met.box.map50:.4f} P={met.box.mp:.4f} R={met.box.mr:.4f}")

    fig,axes=plt.subplots(1,2,figsize=(16,6))
    confs=[r['conf'] for r in thresh_res]; precs=[r['prec'] for r in thresh_res]
    recs=[r['rec'] for r in thresh_res]; maps=[r['map50'] for r in thresh_res]
    axes[0].plot(confs,precs,'b-o',ms=5,label='Precision')
    axes[0].plot(confs,recs,'r-o',ms=5,label='Recall')
    axes[0].plot(confs,maps,'g-o',ms=5,label='mAP@50')
    axes[0].axvline(0.25,color='black',ls='--',label='Default 0.25')
    axes[0].set_xlabel('Confidence Threshold'); axes[0].set_ylabel('Score'); axes[0].set_ylim(0,1)
    axes[0].set_title('Metrics vs Confidence Threshold',fontweight='bold')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(recs,precs,'b-o',ms=5)
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)
    axes[1].set_title('Precision-Recall Curve (M3 Best)',fontweight='bold'); axes[1].grid(alpha=0.3)
    plt.suptitle('Threshold Analysis - Task 6',fontsize=12,fontweight='bold')
    plt.tight_layout(); plt.savefig(M3_YOLO_DIR+"/plots/threshold_analysis.png",dpi=100); plt.close()
    print("Threshold analysis saved")

# Failure case breakdown
t_list=os.listdir(TEST_IMG_DIR); samp=random.sample(t_list,min(50,len(t_list)))
no_det,low_det,high_det=[],[],[]
for fname in samp:
    ip=os.path.join(TEST_IMG_DIR,fname)
    res=det_m3.predict(ip,conf=0.25,verbose=False)[0]
    nb=len(res.boxes) if res.boxes else 0; vis=res.plot()
    if nb==0:  no_det.append((vis,fname,nb))
    elif nb>5: high_det.append((vis,fname,nb))
    else:      low_det.append((vis,fname,nb))
print(f"\n  No detections  : {len(no_det)}/{len(samp)}")
print(f"  1-5 detections : {len(low_det)}/{len(samp)}")
print(f"  6+ detections  : {len(high_det)}/{len(samp)} (possible over-detection)")

if no_det:  det_grid(no_det[:8],'No Detections - Failure Cases','error_no_detections.png')
if high_det:det_grid(high_det[:8],'Over-Detection - 6+ Boxes (possible FP)','error_over_detection.png')

err_summary=dict(tested=len(samp),no_detect=len(no_det),normal_det=len(low_det),
                 over_detect=len(high_det),best_model=best_det_name)
with open(M3_YOLO_DIR+"/error_summary.json",'w') as f: json.dump(err_summary,f,indent=2)
print("Error summary saved")


##  Detection | Final Report

In [ ]:
exp_str=pd.DataFrame(yolo_results)[['name','model','map50','map95','prec','rec','vs_baseline']].to_string(index=False) if yolo_results else "No experiments run yet"

lines=[
"CliniScan - Milestone 3  Detection Report",
"="*45,
f"Generated: {time.strftime('%Y-%m-%d %H:%M')}",
"",
"BASELINE (Milestone 2 - YOLOv8s 30 epochs)",
f"  mAP@50    : {DET_BASELINE['map50']}",
f"  mAP@50-95 : {DET_BASELINE['map50_95']}",
f"  Precision : {DET_BASELINE['precision']}",
f"  Recall    : {DET_BASELINE['recall']}",
"",
"M3 EXPERIMENTS (Task 1 + 3)",
exp_str,
"",
"  D1_YOLOv8m   : Larger YOLOv8m backbone (Task 3)",
"  D2_LowLR     : LR 5e-4 - finer gradient steps (Task 1)",
"  D3_LowThresh : conf=0.15 / iou=0.45 - catches more lesions (Task 1)",
"  D4_SGD       : SGD vs AdamW comparison (Task 1)",
"",
"AUGMENTATIONS (Task 2 - medically safe)",
"  YES: HFlip(0.5), Rotation+-10, Scale+-10%, Brightness",
"  NO : VertFlip, Mosaic, Mixup, Perspective, Color shift",
"",
"DELIVERABLES",
"  milestone3_det/correct_detections.png",
"  milestone3_det/missed_detections.png",
"  milestone3_det/error_no_detections.png",
"  milestone3_det/error_over_detection.png",
"  milestone3_det/plots/det_comparison_bar.png",
"  milestone3_det/plots/m2_vs_m3.png",
"  milestone3_det/plots/threshold_analysis.png",
"  milestone3_det/det_results.csv",
"  milestone3_det/error_summary.json",
]
report="\n".join(lines)
rp=M3_YOLO_DIR+"/report.txt"
with open(rp,'w') as f: f.write(report)
print(report); print(f"\nReport saved: {rp}")
